In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", ".."))  # activate docs/

In [ ]:
using LinearAlgebra, Serialization, Qritical

DATA_ROOT = normpath(joinpath(@__DIR__, "..", "..", "..", "..", "data"))

load_mps(fname, D=64) = let
    ψ_raw = deserialize(joinpath(DATA_ROOT, fname))
    N = ndims(ψ_raw);  d = size(ψ_raw, 1)
    sites = Tuple([upper(Symbol(:s, i), d) for i in 1:N])
    to_mps(QTensor(ψ_raw, sites); trunc=MaxBondDimTrunc(D), form=:left)
end

mps1 = load_mps("psi1.jls")
mps2 = load_mps("psi2.jls")
N    = length(mps1.tensors)
println("Loaded two MPS of length ", N)

# Ex 4. Overlap, Observables, and MPS Addition

## (a) MPS overlap $\langle \psi_1 | \psi_2 \rangle$

The overlap contracts the bra and ket tensors site by site through a
$\chi^2$-dimensional transfer matrix — cost $O(L\chi^3 d)$, exponentially
cheaper than the $O(d^L)$ full-vector dot product.

In [ ]:
o11 = overlap(mps1, mps1)
o22 = overlap(mps2, mps2)
o12 = overlap(mps1, mps2)

println("⟨ψ₁|ψ₁⟩ = ", round(real(o11); sigdigits=8))
println("⟨ψ₂|ψ₂⟩ = ", round(real(o22); sigdigits=8))
println("⟨ψ₁|ψ₂⟩ = ", round(o12;     sigdigits=6))

In [ ]:
# Sanity: self-overlap of a left-canonical MPS is 1 by construction
@assert abs(real(o11) - 1.0) < 1e-12 "‖ψ₁‖ ≠ 1: $o11"
@assert abs(real(o22) - 1.0) < 1e-12 "‖ψ₂‖ ≠ 1: $o22"

# Cauchy-Schwarz: |⟨ψ₁|ψ₂⟩| ≤ 1
@assert abs(o12) ≤ 1.0 + 1e-10  "Cauchy-Schwarz violated"
println("All overlap checks passed ✓")

## (b) Local observables $\langle \sigma^z_i \rangle$ and $\langle \sigma^x_i \rangle$

`local_expectation(mps, op, site)` performs a single left-to-right environment
sweep inserting `op` at `site`.  For a left-canonical MPS the left environment
collapse is automatic, but the function works on any canonical form.

In [ ]:
ops  = algebra_generators(SpinHalf())
σz   = ops.Sz * 2   # σᶻ = 2Sᶻ ∈ {+1, −1}
σx   = (ops.Sp + ops.Sm)   # σˣ = S⁺ + S⁻

σz_vals = [real(local_expectation(mps1, σz, i)) for i in 1:N]
σx_vals = [real(local_expectation(mps1, σx, i)) for i in 1:N]

println("⟨σᶻᵢ⟩: ", round.(σz_vals; sigdigits=3))
println("⟨σˣᵢ⟩: ", round.(σx_vals; sigdigits=3))

In [ ]:
# Cross-check: put center at each site via BondCanonical and recompute ⟨σᶻ⟩
# The value must be gauge-independent
println("⟨σᶻ₁⟩ computed with center at different bonds:")
for l in 1:N
    mps_c = canonicalize(mps1, BondCanonical(l))
    val   = real(local_expectation(mps_c, σz, 1))
    println("  bond-center=$l: ", round(val; sigdigits=6))
end

## (c) Two-site correlations $\langle \sigma^z_i \sigma^z_j \rangle$

`two_site_op(mps, op_i, op_j, i, j)` inserts two operators during a single sweep —
same $O(L\chi^2 d)$ cost as one local measurement.
The *connected* correlator subtracts the product of single-site means.

In [ ]:
# Nearest-neighbour ⟨σᶻᵢ σᶻᵢ₊₁⟩
zz_nn = [real(two_site_op(mps1, σz, σz, i, i+1)) for i in 1:N-1]
println("⟨σᶻᵢ σᶻᵢ₊₁⟩: ", round.(zz_nn; sigdigits=3))

# Connected correlator ⟨σᶻᵢ σᶻᵢ₊₁⟩_c = ⟨σᶻᵢ σᶻᵢ₊₁⟩ − ⟨σᶻᵢ⟩⟨σᶻᵢ₊₁⟩
zz_c = zz_nn .- σz_vals[1:end-1] .* σz_vals[2:end]
println("Connected: ", round.(zz_c; sigdigits=3))

## (d) MPS addition $a|\psi_1\rangle + b|\psi_2\rangle$

`add_mps(a, ψ₁, b, ψ₂)` assembles a block-diagonal MPS of bond dimension
$\chi_1 + \chi_2$, then recompresses via a left sweep.
Linearity is verified via the overlap identity
$\langle\psi_1 | a\psi_1 + b\psi_2\rangle = a\langle\psi_1|\psi_1\rangle + b\langle\psi_1|\psi_2\rangle$.

In [ ]:
a, b = 0.6 + 0.0im, 0.8 + 0.0im
mps_sum = add_mps(a, mps1, b, mps2)

println("Bond dims of sum MPS: ", [size(t.data, 3) for t in mps_sum.tensors])
println("⟨ψ_sum|ψ_sum⟩ = ", round(real(overlap(mps_sum, mps_sum)); sigdigits=8))

In [ ]:
# Linearity check: ⟨ψ₁|aψ₁+bψ₂⟩ = a⟨ψ₁|ψ₁⟩ + b⟨ψ₁|ψ₂⟩
expected = a * o11 + b * o12
computed = overlap(mps1, mps_sum)
println("Expected ⟨ψ₁|aψ₁+bψ₂⟩ = ", round(expected; sigdigits=6))
println("Computed              = ", round(computed; sigdigits=6))
println("Error: ", round(abs(expected - computed); sigdigits=3))

In [ ]:
# Compressed addition: cap bond dimension at D_max
D_max    = 8
mps_comp = add_mps(a, mps1, b, mps2; trunc=MaxBondDimTrunc(D_max))
println("Compressed bond dims (D_max=$D_max): ", [size(t.data, 3) for t in mps_comp.tensors])
# Overlap with exact sum measures truncation fidelity
fidelity = abs(overlap(mps_comp, mps_sum))^2 / (real(overlap(mps_comp, mps_comp)) * real(overlap(mps_sum, mps_sum)))
println("Fidelity |⟨compressed|exact⟩|² = ", round(fidelity; sigdigits=5))